In [1]:
import sys
sys.path.append("..")          # lets the notebook find files in src/
 
import numpy as np
import pandas as pd
from src.db import get_engine
 
engine = get_engine()


In [15]:
rng = np.random.default_rng(42)
states = ["Selangor", "Johor", "Penang", "Sabah", "Sarawak", "Kelantan"]
categories = ["Fiction", "Business", "Technology", "Children"]
n = 500

sales = pd.DataFrame({
    "sale_date": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "state": rng.choice(states, size=n),
    "category": rng.choice(categories, size=n),
    "amount_myr": rng.gamma(2.0, 40.0, n).round(2),
})

targets = pd.DataFrame({"state": states, "q1_target_myr": [9000, 6000, 5000, 3000, 3500, 2500]})
sales.head()


,sale_date,state,category,amount_myr
0,2026-01-09,Selangor,Children,61.82
1,2026-03-11,Sabah,Technology,4.63
2,2026-02-28,Kelantan,Technology,29.46
3,2026-02-09,Kelantan,Children,82.35
4,2026-02-08,Johor,Fiction,23.42


In [16]:
sales.to_sql("practice_sales", engine, if_exists="replace", index=False)
targets.to_sql("practice_targets", engine, if_exists="replace", index=False)
print("Loaded", len(sales), "sales rows and", len(targets), "targets")


Loaded 500 sales rows and 6 targets


In [17]:
def q(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql(sql, engine)
 
q("SELECT COUNT(*) AS row_count FROM practice_sales")


,row_count
0,500


In [18]:
q("""
SELECT state, COUNT(*) AS orders, ROUND(SUM(amount_myr)::numeric, 2) AS revenue
FROM practice_sales
GROUP BY state
ORDER BY revenue DESC
""")


,state,orders,revenue
0,Selangor,93,7732.62
1,Sabah,96,7341.15
2,Kelantan,86,6441.66
3,Sarawak,83,6411.48
4,Johor,72,5920.78
5,Penang,70,4986.58


In [19]:
q("""
SELECT s.state,
       ROUND(SUM(s.amount_myr)::numeric, 2) AS revenue,
       t.q1_target_myr,
       ROUND((SUM(s.amount_myr) / t.q1_target_myr * 100)::numeric, 1) AS pct_of_target
FROM practice_sales s
JOIN practice_targets t ON s.state = t.state
GROUP BY s.state, t.q1_target_myr
""")


,state,revenue,q1_target_myr,pct_of_target
0,Penang,4986.58,5000,99.7
1,Selangor,7732.62,9000,85.9
2,Sabah,7341.15,3000,244.7
3,Kelantan,6441.66,2500,257.7
4,Sarawak,6411.48,3500,183.2
5,Johor,5920.78,6000,98.7


In [20]:
# Question 3
q("""
SELECT category, ROUND(AVG(amount_myr)::numeric, 2) AS avg_order
FROM practice_sales
GROUP BY category
ORDER BY avg_order DESC
""")


,category,avg_order
0,Fiction,84.55
1,Business,77.01
2,Technology,75.13
3,Children,74.72


In [21]:
# Question 4
q("""
SELECT DATE(sale_date), SUM(amount_myr) AS daily_sales
FROM practice_sales
GROUP BY DATE(sale_date)
ORDER BY DATE(sale_date)
""")


,date,daily_sales
0,2026-01-01,196.26
1,2026-01-02,531.09
2,2026-01-03,362.22
3,2026-01-04,359.41
4,2026-01-05,133.17
...,...,...
85,2026-03-27,246.91
86,2026-03-28,468.44
87,2026-03-29,866.08
88,2026-03-30,66.82


In [22]:
# Question 5
q("""
SELECT state, category, SUM(amount_myr) AS total
FROM practice_sales
GROUP BY ROLLUP(state, category)
ORDER BY state, category
""")


,state,category,total
0,Johor,Business,1907.24
1,Johor,Children,1085.64
2,Johor,Fiction,983.86
3,Johor,Technology,1944.04
4,Johor,None,5920.78
5,Kelantan,Business,1860.03
6,Kelantan,Children,1841.46
7,Kelantan,Fiction,1420.00
8,Kelantan,Technology,1320.17
9,Kelantan,None,6441.66


In [24]:
df_q1_pandas = sales.groupby("state").agg(
    orders=("amount_myr", "size"),
    revenue=("amount_myr", "sum")
).sort_values("revenue", ascending=False).reset_index()

df_q1_pandas["revenue"] = df_q1_pandas["revenue"].round(2)
df_q1_pandas


,state,orders,revenue
0,Selangor,93,7732.62
1,Sabah,96,7341.15
2,Kelantan,86,6441.66
3,Sarawak,83,6411.48
4,Johor,72,5920.78
5,Penang,70,4986.58


In [25]:
q("""
WITH ranked AS (
    SELECT
        state,
        category,
        SUM(amount_myr) AS total,
        RANK() OVER (PARTITION BY state ORDER BY SUM(amount_myr) DESC) AS rank_in_state
    FROM practice_sales
    GROUP BY state, category
)
SELECT state, category, total
FROM ranked
WHERE rank_in_state = 1
ORDER BY state;
""")


,state,category,total
0,Johor,Technology,1944.04
1,Kelantan,Business,1860.03
2,Penang,Fiction,1595.48
3,Sabah,Children,2155.13
4,Sarawak,Fiction,2277.30
5,Selangor,Children,2462.26


In [26]:
q("""
SELECT
    sale_date,
    daily_sales,
    AVG(daily_sales) OVER (
        ORDER BY sale_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS seven_day_avg
FROM (
    SELECT DATE(sale_date) AS sale_date, SUM(amount_myr) AS daily_sales
    FROM practice_sales
    GROUP BY DATE(sale_date)
) t
ORDER BY sale_date;
""")


,sale_date,daily_sales,seven_day_avg
0,2026-01-01,196.26,196.260000
1,2026-01-02,531.09,363.675000
2,2026-01-03,362.22,363.190000
3,2026-01-04,359.41,362.245000
4,2026-01-05,133.17,316.430000
...,...,...,...
85,2026-03-27,246.91,381.688571
86,2026-03-28,468.44,430.732857
87,2026-03-29,866.08,483.494286
88,2026-03-30,66.82,437.720000
